In [1]:
!pip install transformers

import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, f1_score

from transformers import AutoTokenizer, AutoModel, ViTModel
import torchvision.models as models

In [6]:
import os

print(os.listdir("/kaggle/input/datasets/rranjithkumariiitk/train-text"))
print(os.listdir("/kaggle/input/datasets/rranjithkumariiitk/dev-text"))
print(os.listdir("/kaggle/input/datasets/rranjithkumariiitk/test-dataset"))

['train (1).csv']
['dev (1).csv']
['test.csv']


In [9]:
train_df = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/train-text/train (1).csv")
dev_df   = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/dev-text/dev (1).csv")
test_df  = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/test-dataset/test.csv")

In [35]:
import os

class MemeDataset(Dataset):
    def __init__(self, df, image_path, is_test=False):
        self.df = df
        self.image_path = image_path
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row["transcriptions"])

        # 🔥 try multiple formats safely
        img_name = str(row["image_id"])

        possible_paths = [
            f"{self.image_path}/{img_name}.jpg",
            f"{self.image_path}/{img_name}.png",
            f"{self.image_path}/{img_name}.jpeg"
        ]

        img_path = None
        for p in possible_paths:
            if os.path.exists(p):
                img_path = p
                break

        # 🚨 If still not found → raise clear error
        if img_path is None:
            raise FileNotFoundError(f"Image not found for ID: {img_name}")

        image = Image.open(img_path).convert("RGB")

        muril_enc = muril_tokenizer(text, padding='max_length',
                                   truncation=True, max_length=128,
                                   return_tensors="pt")

        xlmr_enc = xlmr_tokenizer(text, padding='max_length',
                                 truncation=True, max_length=128,
                                 return_tensors="pt")

        item = {
            "input_ids1": muril_enc["input_ids"].squeeze(0),
            "attention_mask1": muril_enc["attention_mask"].squeeze(0),
            "input_ids2": xlmr_enc["input_ids"].squeeze(0),
            "attention_mask2": xlmr_enc["attention_mask"].squeeze(0),
            "image": transform(image)
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.long)

        return item

In [54]:
train_loader = DataLoader(
    MemeDataset(train_df, "/kaggle/input/datasets/rranjithkumariiitk/train-image"),
    batch_size=4, shuffle=True
)

dev_loader = DataLoader(
    MemeDataset(dev_df, "/kaggle/input/datasets/rranjithkumariiitk/dev-image"),
    batch_size=4
)

test_loader = DataLoader(
    MemeDataset(test_df, "/kaggle/input/datasets/rranjithkumariiitk/test-image", is_test=True),
    batch_size=4
)

In [12]:
sample = train_df.iloc[0]['image_id']
path = f"/kaggle/input/datasets/rranjithkumariiitk/train-image/{sample}.jpg"

print("Path:", path)
print("Exists:", os.path.exists(path))

Path: /kaggle/input/datasets/rranjithkumariiitk/train-image/391.jpg
Exists: False


In [13]:
img_path = f"{self.image_path}/{row['image_id']}"

NameError: name 'self' is not defined

In [14]:
print(train_df.columns)

Index(['image_id', 'transcriptions', 'original_labels', 'irish_labels',
       'chinese_labels'],
      dtype='object')


In [15]:
row["transcriptions"]
row["image_id"]
row["original_labels"]

NameError: name 'row' is not defined

In [16]:
label_map = {"not-misogyny": 0, "misogyny": 1}

train_df["label"] = train_df["original_labels"].map(label_map)
dev_df["label"]   = dev_df["original_labels"].map(label_map)

In [18]:
muril_tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
xlmr_tokenizer  = AutoTokenizer.from_pretrained("xlm-roberta-base")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [29]:
import torch.nn as nn
from transformers import AutoModel

class TextModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.muril = AutoModel.from_pretrained("google/muril-base-cased")
        self.fc = nn.Linear(768, 2)

    def forward(self, ids, mask):
        x = self.muril(input_ids=ids, attention_mask=mask)
        return self.fc(x.last_hidden_state[:,0,:])

In [30]:
import torchvision.models as models

class ImageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Linear(2048, 2)

    def forward(self, img):
        return self.resnet(img)

In [31]:
from transformers import ViTModel

class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.muril = AutoModel.from_pretrained("google/muril-base-cased")
        self.xlmr  = AutoModel.from_pretrained("xlm-roberta-base")

        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()

        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")

        self.text_proj  = nn.Linear(768*2, 512)
        self.image_proj = nn.Linear(2048+768, 512)

        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 2)
        )

    def forward(self, i1, m1, i2, m2, img):
        t1 = self.muril(input_ids=i1, attention_mask=m1).last_hidden_state[:,0,:]
        t2 = self.xlmr(input_ids=i2, attention_mask=m2).last_hidden_state[:,0,:]

        text = self.text_proj(torch.cat((t1,t2), dim=1))

        r = self.resnet(img)
        v = self.vit(pixel_values=img).last_hidden_state[:,0,:]

        image = self.image_proj(torch.cat((r,v), dim=1))

        return self.classifier(torch.cat((text,image), dim=1))

In [59]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text_model  = TextModel().to(device)
image_model = ImageModel().to(device)
multi_model = MultiModalModel().to(device)

criterion = nn.CrossEntropyLoss()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OutOfMemoryError: CUDA out of memory. Tried to allocate 578.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 233.81 MiB is free. Including non-PyTorch memory, this process has 14.33 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 1.16 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [33]:
def train_model(model, mode):
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

    for epoch in range(3):
        model.train()
        total_loss = 0

        for b in train_loader:

            if mode == "text":
                out = model(
                    b['input_ids1'].to(device),
                    b['attention_mask1'].to(device)
                )

            elif mode == "image":
                out = model(b['image'].to(device))

            else:
                out = model(
                    b['input_ids1'].to(device),
                    b['attention_mask1'].to(device),
                    b['input_ids2'].to(device),
                    b['attention_mask2'].to(device),
                    b['image'].to(device)
                )

            loss = criterion(out, b['label'].to(device))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"{mode} Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

In [ ]:
train_model(text_model, "text")
train_model(image_model, "image")
train_model(multi_model, "multi")

In [57]:
def train_model(model, mode):
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

    for epoch in range(3):
        model.train()
        for b in train_loader:

            if mode == "text":
                out = model(b['input_ids1'].to(device),
                            b['attention_mask1'].to(device))

            elif mode == "image":
                out = model(b['image'].to(device))

            else:
                out = model(
                    b['input_ids1'].to(device),
                    b['attention_mask1'].to(device),
                    b['input_ids2'].to(device),
                    b['attention_mask2'].to(device),
                    b['image'].to(device)
                )

            loss = criterion(out, b['label'].to(device))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [58]:
train_model(text_model, "text")
train_model(image_model, "image")
train_model(multi_model, "multi")

print(train_df.iloc[0]['image_id'])
print(train_df.iloc[100]['image_id'])

import os

files = os.listdir("/kaggle/input/datasets/rranjithkumariiitk/train-image")
print(files[:20])

TRAIN_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/train-image/train"

DEV_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/dev-image/dev"

TEST_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/test-image/test"

train_loader = DataLoader(
    MemeDataset(train_df, TRAIN_IMG_PATH),
    batch_size=4, shuffle=True
)

dev_loader = DataLoader(
    MemeDataset(dev_df, DEV_IMG_PATH),
    batch_size=4
)

test_loader = DataLoader(
    MemeDataset(test_df, TEST_IMG_PATH, is_test=True),
    batch_size=4
)

import os

files = os.listdir("/kaggle/input/datasets/rranjithkumariiitk/train-image/train")
print(files[:10])

FileNotFoundError: Image not found for ID: 808

In [48]:
class TextModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.muril = AutoModel.from_pretrained("google/muril-base-cased")
        self.fc = nn.Linear(768, 2)

    def forward(self, ids, mask):
        x = self.muril(input_ids=ids, attention_mask=mask)
        return self.fc(x.last_hidden_state[:,0,:])

In [49]:
class ImageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Linear(2048, 2)

    def forward(self, img):
        return self.resnet(img)

In [50]:
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.muril = AutoModel.from_pretrained("google/muril-base-cased")
        self.xlmr  = AutoModel.from_pretrained("xlm-roberta-base")

        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()

        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")

        self.text_proj  = nn.Linear(768*2, 512)
        self.image_proj = nn.Linear(2048+768, 512)

        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 2)
        )

    def forward(self, i1, m1, i2, m2, img):
        t1 = self.muril(input_ids=i1, attention_mask=m1).last_hidden_state[:,0,:]
        t2 = self.xlmr(input_ids=i2, attention_mask=m2).last_hidden_state[:,0,:]

        text = self.text_proj(torch.cat((t1,t2), dim=1))

        r = self.resnet(img)
        v = self.vit(pixel_values=img).last_hidden_state[:,0,:]

        image = self.image_proj(torch.cat((r,v), dim=1))

        return self.classifier(torch.cat((text,image), dim=1))

In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text_model  = TextModel().to(device)
image_model = ImageModel().to(device)
multi_model = MultiModalModel().to(device)

criterion = nn.CrossEntropyLoss()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [52]:
def train_model(model, mode):
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

    for epoch in range(3):
        model.train()

        for b in train_loader:

            if mode == "text":
                out = model(b['input_ids1'].to(device),
                            b['attention_mask1'].to(device))

            elif mode == "image":
                out = model(b['image'].to(device))

            else:
                out = model(
                    b['input_ids1'].to(device),
                    b['attention_mask1'].to(device),
                    b['input_ids2'].to(device),
                    b['attention_mask2'].to(device),
                    b['image'].to(device)
                )

            loss = criterion(out, b['label'].to(device))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [55]:
train_model(text_model, "text")
train_model(image_model, "image")
train_model(multi_model, "multi")

FileNotFoundError: Image not found for ID: 1

In [61]:
# ===================== INSTALL =====================
!pip install transformers -q

# ===================== IMPORTS =====================
import torch
import torch.nn as nn
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, ViTModel
import torchvision.models as models

# ===================== PATHS =====================
TRAIN_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/train-image/train"
DEV_IMG_PATH   = "/kaggle/input/datasets/rranjithkumariiitk/dev-image/dev"
TEST_IMG_PATH  = "/kaggle/input/datasets/rranjithkumariiitk/test-image/test"

train_df = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/train-text/train (1).csv")
dev_df   = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/dev-text/dev (1).csv")
test_df  = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/test-dataset/test.csv")

# ===================== LABEL =====================
label_map = {"not-misogyny": 0, "misogyny": 1}
train_df["label"] = train_df["original_labels"].map(label_map)
dev_df["label"]   = dev_df["original_labels"].map(label_map)

# ===================== TOKENIZER =====================
muril_tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# ===================== TRANSFORM =====================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ===================== DATASET =====================
class MemeDataset(Dataset):
    def __init__(self, df, img_path, is_test=False):
        self.df = df
        self.img_path = img_path
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["transcriptions"])
        img_id = str(row["image_id"])

        # safe image loading
        img_path = f"{self.img_path}/{img_id}.jpg"
        if not os.path.exists(img_path):
            img_path = img_path.replace(".jpg", ".png")

        if os.path.exists(img_path):
            image = Image.open(img_path).convert("RGB")
            image = transform(image)
        else:
            image = torch.zeros(3,224,224)

        enc = muril_tokenizer(text, padding='max_length',
                              truncation=True, max_length=64,
                              return_tensors="pt")

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image": image
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.long)

        return item

# ===================== LOADERS =====================
train_loader = DataLoader(MemeDataset(train_df, TRAIN_IMG_PATH), batch_size=2, shuffle=True)
dev_loader   = DataLoader(MemeDataset(dev_df, DEV_IMG_PATH), batch_size=2)
test_loader  = DataLoader(MemeDataset(test_df, TEST_IMG_PATH, True), batch_size=2)

# ===================== MODEL =====================
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_model = AutoModel.from_pretrained("google/muril-base-cased")

        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()

        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")

        self.text_proj  = nn.Linear(768, 256)
        self.image_proj = nn.Linear(2048+768, 256)

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, ids, mask, img):
        t = self.text_model(input_ids=ids, attention_mask=mask).last_hidden_state[:,0,:]
        t = self.text_proj(t)

        r = self.resnet(img)
        v = self.vit(pixel_values=img).last_hidden_state[:,0,:]
        i = self.image_proj(torch.cat((r,v), dim=1))

        return self.classifier(torch.cat((t,i), dim=1))

# ===================== INIT =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalModel().to(device)

# freeze heavy parts
for p in model.text_model.parameters():
    p.requires_grad = False
for p in model.resnet.parameters():
    p.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
scaler = torch.cuda.amp.GradScaler()

# ===================== TRAIN =====================
for epoch in range(3):
    model.train()
    total_loss = 0

    for b in train_loader:
        ids = b['input_ids'].to(device)
        mask = b['attention_mask'].to(device)
        img = b['image'].to(device)
        labels = b['label'].to(device)

        with torch.cuda.amp.autocast():
            out = model(ids, mask, img)
            loss = criterion(out, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

# ===================== EVAL =====================
model.eval()
preds, true = [], []

with torch.no_grad():
    for b in dev_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)
        true.extend(b['label'].numpy())

print("F1:", f1_score(true, preds))

# ===================== TEST =====================
preds = []
with torch.no_grad():
    for b in test_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        preds.extend(torch.argmax(out, dim=1).cpu().numpy())

reverse_map = {0:"not-misogyny", 1:"misogyny"}
test_df["original_labels"] = [reverse_map[p] for p in preds]

test_df.to_csv("submission.csv", index=False)

print("✅ DONE — Submission file created")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarni

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OutOfMemoryError: CUDA out of memory. Tried to allocate 578.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 233.81 MiB is free. Including non-PyTorch memory, this process has 14.33 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 1.16 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [62]:
self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")

# keep only first 6 layers
self.vit.encoder.layer = self.vit.encoder.layer[:6]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'self' is not defined

In [64]:
# ===================== INSTALL =====================
!pip install transformers -q

# ===================== IMPORTS =====================
import torch
import torch.nn as nn
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, ViTModel
import torchvision.models as models

# ===================== MEMORY FIX =====================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ===================== PATHS =====================
TRAIN_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/train-image/train"
DEV_IMG_PATH   = "/kaggle/input/datasets/rranjithkumariiitk/dev-image/dev"
TEST_IMG_PATH  = "/kaggle/input/datasets/rranjithkumariiitk/test-image/test"

train_df = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/train-text/train (1).csv")
dev_df   = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/dev-text/dev (1).csv")
test_df  = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/test-dataset/test.csv")

# ===================== LABEL =====================
label_map = {"not-misogyny": 0, "misogyny": 1}
train_df["label"] = train_df["original_labels"].map(label_map)
dev_df["label"]   = dev_df["original_labels"].map(label_map)

# ===================== TOKENIZER =====================
muril_tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# ===================== TRANSFORM =====================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ===================== DATASET =====================
class MemeDataset(Dataset):
    def __init__(self, df, img_path, is_test=False):
        self.df = df
        self.img_path = img_path
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["transcriptions"])
        img_id = str(row["image_id"])

        img_path = f"{self.img_path}/{img_id}.jpg"
        if not os.path.exists(img_path):
            img_path = img_path.replace(".jpg", ".png")

        if os.path.exists(img_path):
            image = Image.open(img_path).convert("RGB")
            image = transform(image)
        else:
            image = torch.zeros(3,224,224)

        enc = muril_tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=32,   # 🔥 reduced
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image": image
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.long)

        return item

# ===================== LOADERS =====================
train_loader = DataLoader(MemeDataset(train_df, TRAIN_IMG_PATH), batch_size=1, shuffle=True)
dev_loader   = DataLoader(MemeDataset(dev_df, DEV_IMG_PATH), batch_size=1)
test_loader  = DataLoader(MemeDataset(test_df, TEST_IMG_PATH, True), batch_size=1)

# ===================== MODEL =====================
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_model = AutoModel.from_pretrained("google/muril-base-cased")

        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()

        # ✅ KEEP ViT but LIGHTEN IT
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        self.vit.encoder.layer = self.vit.encoder.layer[:6]

        # freeze heavy parts
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.resnet.parameters():
            p.requires_grad = False
        for p in self.vit.parameters():
            p.requires_grad = False

        self.text_proj  = nn.Linear(768, 256)
        self.image_proj = nn.Linear(2048+768, 256)

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, ids, mask, img):

        t = self.text_model(input_ids=ids, attention_mask=mask).last_hidden_state[:,0,:]
        t = self.text_proj(t)

        r = self.resnet(img)

        # ✅ HUGE MEMORY SAVE
        with torch.no_grad():
            v = self.vit(pixel_values=img).last_hidden_state[:,0,:]

        i = self.image_proj(torch.cat((r,v), dim=1))

        return self.classifier(torch.cat((t,i), dim=1))

# ===================== INIT =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalModel().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
scaler = torch.cuda.amp.GradScaler()

# ===================== TRAIN =====================
for epoch in range(3):
    model.train()
    total_loss = 0

    for b in train_loader:
        ids = b['input_ids'].to(device)
        mask = b['attention_mask'].to(device)
        img = b['image'].to(device)
        labels = b['label'].to(device)

        with torch.cuda.amp.autocast():
            out = model(ids, mask, img)
            loss = criterion(out, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        torch.cuda.empty_cache()  # 🔥 prevent fragmentation

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

# ===================== EVAL =====================
model.eval()
preds, true = [], []

with torch.no_grad():
    for b in dev_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)
        true.extend(b['label'].numpy())

print("F1:", f1_score(true, preds))

# ===================== TEST =====================
preds = []
with torch.no_grad():
    for b in test_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        preds.extend(torch.argmax(out, dim=1).cpu().numpy())

reverse_map = {0:"not-misogyny", 1:"misogyny"}
test_df["original_labels"] = [reverse_map[p] for p in preds]

test_df.to_csv("submission.csv", index=False)

print("✅ DONE — Submission file created")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarni

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OutOfMemoryError: CUDA out of memory. Tried to allocate 578.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 233.81 MiB is free. Including non-PyTorch memory, this process has 14.33 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 1.16 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [65]:
# ===================== INSTALL =====================
!pip install transformers -q

# ===================== IMPORTS =====================
import torch
import torch.nn as nn
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, ViTModel
import torchvision.models as models

# ===================== MEMORY FIX =====================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ===================== PATHS =====================
TRAIN_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/train-image/train"
DEV_IMG_PATH   = "/kaggle/input/datasets/rranjithkumariiitk/dev-image/dev"
TEST_IMG_PATH  = "/kaggle/input/datasets/rranjithkumariiitk/test-image/test"

train_df = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/train-text/train (1).csv")
dev_df   = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/dev-text/dev (1).csv")
test_df  = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/test-dataset/test.csv")

# ===================== LABEL =====================
label_map = {"not-misogyny": 0, "misogyny": 1}
train_df["label"] = train_df["original_labels"].map(label_map)
dev_df["label"]   = dev_df["original_labels"].map(label_map)

# ===================== TOKENIZER =====================
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# ===================== TRANSFORM =====================
transform = transforms.Compose([
    transforms.Resize((192,192)),  # reduced
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ===================== DATASET =====================
class MemeDataset(Dataset):
    def __init__(self, df, img_path, is_test=False):
        self.df = df
        self.img_path = img_path
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["transcriptions"])
        img_id = str(row["image_id"])

        img_path = f"{self.img_path}/{img_id}.jpg"
        if not os.path.exists(img_path):
            img_path = img_path.replace(".jpg", ".png")

        if os.path.exists(img_path):
            image = Image.open(img_path).convert("RGB")
            image = transform(image)
        else:
            image = torch.zeros(3,192,192)

        enc = tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=24,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image": image
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.long)

        return item

# ===================== LOADERS =====================
train_loader = DataLoader(MemeDataset(train_df, TRAIN_IMG_PATH), batch_size=1, shuffle=True)
dev_loader   = DataLoader(MemeDataset(dev_df, DEV_IMG_PATH), batch_size=1)
test_loader  = DataLoader(MemeDataset(test_df, TEST_IMG_PATH, True), batch_size=1)

# ===================== MODEL =====================
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_model = AutoModel.from_pretrained("google/muril-base-cased")

        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()

        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        self.vit.encoder.layer = self.vit.encoder.layer[:4]

        # freeze all heavy parts
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.resnet.parameters():
            p.requires_grad = False
        for p in self.vit.parameters():
            p.requires_grad = False

        self.text_proj  = nn.Linear(768, 256)
        self.image_proj = nn.Linear(2048+768, 256)

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, ids, mask, img):

        device = ids.device

        # ---- TEXT ----
        with torch.no_grad():
            t = self.text_model(input_ids=ids, attention_mask=mask).last_hidden_state[:,0,:]
        t = t.detach()
        torch.cuda.empty_cache()

        # ---- RESNET ----
        with torch.no_grad():
            r = self.resnet(img)
        r = r.detach()
        torch.cuda.empty_cache()

        # ---- VIT (move temporarily to GPU) ----
        self.vit.to(device)
        with torch.no_grad():
            v = self.vit(pixel_values=img).last_hidden_state[:,0,:]
        v = v.detach()

        # move vit back to CPU
        self.vit.to("cpu")
        torch.cuda.empty_cache()

        # ---- PROJECTION ----
        t = self.text_proj(t)
        i = self.image_proj(torch.cat((r, v.to(device)), dim=1))

        return self.classifier(torch.cat((t, i), dim=1))

# ===================== INIT =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalModel().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
scaler = torch.cuda.amp.GradScaler()

# ===================== TRAIN =====================
for epoch in range(3):
    model.train()
    total_loss = 0

    for b in train_loader:
        ids = b['input_ids'].to(device)
        mask = b['attention_mask'].to(device)
        img = b['image'].to(device)
        labels = b['label'].to(device)

        with torch.cuda.amp.autocast():
            out = model(ids, mask, img)
            loss = criterion(out, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        torch.cuda.empty_cache()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

# ===================== EVAL =====================
model.eval()
preds, true = [], []

with torch.no_grad():
    for b in dev_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)
        true.extend(b['label'].numpy())

print("F1:", f1_score(true, preds))

# ===================== TEST =====================
preds = []
with torch.no_grad():
    for b in test_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        preds.extend(torch.argmax(out, dim=1).cpu().numpy())

reverse_map = {0:"not-misogyny", 1:"misogyny"}
test_df["original_labels"] = [reverse_map[p] for p in preds]

test_df.to_csv("submission.csv", index=False)

print("✅ DONE — Submission file created")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarni

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OutOfMemoryError: CUDA out of memory. Tried to allocate 578.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 233.81 MiB is free. Including non-PyTorch memory, this process has 14.33 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 1.16 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [66]:
# ===================== INSTALL =====================
!pip install transformers -q

# ===================== IMPORTS =====================
import torch
import torch.nn as nn
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as transforms
from transformers import AutoTokenizer, AutoModel, ViTModel
import torchvision.models as models
from sklearn.metrics import f1_score

# ===================== DEVICE =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===================== PATHS =====================
TRAIN_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/train-image/train"
DEV_IMG_PATH   = "/kaggle/input/datasets/rranjithkumariiitk/dev-image/dev"
TEST_IMG_PATH  = "/kaggle/input/datasets/rranjithkumariiitk/test-image/test"

train_df = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/train-text/train (1).csv")
dev_df   = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/dev-text/dev (1).csv")
test_df  = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/test-dataset/test.csv")

# ===================== LABEL =====================
label_map = {"not-misogyny": 0, "misogyny": 1}
train_df["label"] = train_df["original_labels"].map(label_map)
dev_df["label"]   = dev_df["original_labels"].map(label_map)

# ===================== TOKENIZER =====================
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# ===================== TRANSFORM =====================
transform = transforms.Compose([
    transforms.Resize((192,192)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ===================== LOAD MODELS (ONE BY ONE) =====================
text_model = AutoModel.from_pretrained("google/muril-base-cased").to(device).eval()
resnet = models.resnet50(pretrained=True)
resnet.fc = nn.Identity()
resnet = resnet.to(device).eval()

vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
vit.encoder.layer = vit.encoder.layer[:4]
vit = vit.to(device).eval()

# ===================== FEATURE EXTRACTION =====================
def extract_features(df, img_path):
    features = []

    for i in tqdm(range(len(df))):
        row = df.iloc[i]
        text = str(row["transcriptions"])
        img_id = str(row["image_id"])

        # ---- IMAGE ----
        img_file = f"{img_path}/{img_id}.jpg"
        if not os.path.exists(img_file):
            img_file = img_file.replace(".jpg", ".png")

        if os.path.exists(img_file):
            image = Image.open(img_file).convert("RGB")
            image = transform(image).unsqueeze(0).to(device)
        else:
            image = torch.zeros(1,3,192,192).to(device)

        # ---- TEXT ----
        enc = tokenizer(text, return_tensors="pt",
                        truncation=True, padding="max_length", max_length=24)
        ids = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)

        # ---- TEXT FEATURE ----
        with torch.no_grad():
            t = text_model(ids, attention_mask=mask).last_hidden_state[:,0,:]

        torch.cuda.empty_cache()

        # ---- RESNET FEATURE ----
        with torch.no_grad():
            r = resnet(image)

        torch.cuda.empty_cache()

        # ---- VIT FEATURE ----
        with torch.no_grad():
            v = vit(pixel_values=image).last_hidden_state[:,0,:]

        torch.cuda.empty_cache()

        feat = torch.cat((t, r, v), dim=1).cpu()
        features.append(feat)

    return torch.cat(features)

# ===================== EXTRACT =====================
print("Extracting train features...")
train_features = extract_features(train_df, TRAIN_IMG_PATH)

print("Extracting dev features...")
dev_features = extract_features(dev_df, DEV_IMG_PATH)

# ===================== CLASSIFIER =====================
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768+2048+768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.net(x)

clf = Classifier().to(device)

# ===================== TRAIN =====================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(clf.parameters(), lr=2e-4)

X = train_features.to(device)
y = torch.tensor(train_df["label"].values).to(device)

for epoch in range(5):
    clf.train()
    out = clf(X)
    loss = criterion(out, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

# ===================== EVAL =====================
clf.eval()
with torch.no_grad():
    preds = torch.argmax(clf(dev_features.to(device)), dim=1).cpu().numpy()

f1 = f1_score(dev_df["label"], preds)
print("F1 Score:", f1)

# ===================== TEST =====================
print("Extracting test features...")
test_features = extract_features(test_df, TEST_IMG_PATH)

with torch.no_grad():
    test_preds = torch.argmax(clf(test_features.to(device)), dim=1).cpu().numpy()

reverse_map = {0:"not-misogyny", 1:"misogyny"}
test_df["original_labels"] = [reverse_map[p] for p in test_preds]

test_df.to_csv("submission.csv", index=False)

print("✅ DONE — Submission file created")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OutOfMemoryError: CUDA out of memory. Tried to allocate 578.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 233.81 MiB is free. Including non-PyTorch memory, this process has 14.33 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 1.16 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [1]:
# ===================== INSTALL =====================
!pip install transformers -q

# ===================== IMPORTS =====================
import torch
import torch.nn as nn
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel
import torchvision.models as models

# ===================== MEMORY FIX =====================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ===================== PATHS =====================
TRAIN_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/train-image/train"
DEV_IMG_PATH   = "/kaggle/input/datasets/rranjithkumariiitk/dev-image/dev"
TEST_IMG_PATH  = "/kaggle/input/datasets/rranjithkumariiitk/test-image/test"

train_df = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/train-text/train (1).csv")
dev_df   = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/dev-text/dev (1).csv")
test_df  = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/test-dataset/test.csv")

# ===================== LABEL =====================
label_map = {"not-misogyny": 0, "misogyny": 1}
train_df["label"] = train_df["original_labels"].map(label_map)
dev_df["label"]   = dev_df["original_labels"].map(label_map)

# ===================== TOKENIZER =====================
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# ===================== TRANSFORM =====================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ===================== DATASET =====================
class MemeDataset(Dataset):
    def __init__(self, df, img_path, is_test=False):
        self.df = df
        self.img_path = img_path
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["transcriptions"])
        img_id = str(row["image_id"])

        img_path = f"{self.img_path}/{img_id}.jpg"
        if not os.path.exists(img_path):
            img_path = img_path.replace(".jpg", ".png")

        if os.path.exists(img_path):
            image = Image.open(img_path).convert("RGB")
            image = transform(image)
        else:
            image = torch.zeros(3,224,224)

        enc = tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image": image
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.long)

        return item

# ===================== LOADERS =====================
train_loader = DataLoader(MemeDataset(train_df, TRAIN_IMG_PATH), batch_size=2, shuffle=True)
dev_loader   = DataLoader(MemeDataset(dev_df, DEV_IMG_PATH), batch_size=2)
test_loader  = DataLoader(MemeDataset(test_df, TEST_IMG_PATH, True), batch_size=2)

# ===================== MODEL =====================
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        # TEXT MODEL
        self.text_model = AutoModel.from_pretrained("google/muril-base-cased")

        # IMAGE MODEL (ResNet)
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()

        # Freeze heavy layers
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.resnet.parameters():
            p.requires_grad = False

        # Projection layers
        self.text_proj  = nn.Linear(768, 256)
        self.image_proj = nn.Linear(2048, 256)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, ids, mask, img):

        with torch.no_grad():
            t = self.text_model(input_ids=ids, attention_mask=mask).last_hidden_state[:,0,:]

        with torch.no_grad():
            r = self.resnet(img)

        t = self.text_proj(t)
        i = self.image_proj(r)

        return self.classifier(torch.cat((t,i), dim=1))

# ===================== INIT =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalModel().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
scaler = torch.cuda.amp.GradScaler()

# ===================== TRAIN =====================
for epoch in range(3):
    model.train()
    total_loss = 0

    for b in train_loader:
        ids = b['input_ids'].to(device)
        mask = b['attention_mask'].to(device)
        img = b['image'].to(device)
        labels = b['label'].to(device)

        with torch.cuda.amp.autocast():
            out = model(ids, mask, img)
            loss = criterion(out, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

# ===================== EVAL =====================
model.eval()
preds, true = [], []

with torch.no_grad():
    for b in dev_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)
        true.extend(b['label'].numpy())

print("F1:", f1_score(true, preds))

# ===================== TEST =====================
preds = []
with torch.no_grad():
    for b in test_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        preds.extend(torch.argmax(out, dim=1).cpu().numpy())

reverse_map = {0:"not-misogyny", 1:"misogyny"}
test_df["original_labels"] = [reverse_map[p] for p in preds]

test_df.to_csv("submission.csv", index=False)

print("✅ DONE — Submission file created")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarni

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
 14%|█▍        | 13.9M/97.8M [00:00<00:00, 145MB/s]
 29%|██▉       | 28.2M/97.8M [00:00<00:00, 147MB/s]
 43%|████▎     | 42.4M/97.8M [00:00<00:00, 125MB/s]
 66%|██████▌   | 64.6M/97.8M [00:00<00:00, 163MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 160MB/s]
/tmp/ipykernel_55/1315454802.py:142: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_55/1315454802.py:155: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1, Loss: 0.6749919891357422
Epoch 2, Loss: 0.6652168273925781
Epoch 3, Loss: 0.6512882232666015
F1: 0.0
✅ DONE — Submission file created


In [1]:
# ===================== INSTALL =====================
!pip install transformers -q

# ===================== IMPORTS =====================
import torch
import torch.nn as nn
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel
import torchvision.models as models
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# ===================== MEMORY CLEAN =====================
import gc
gc.collect()
torch.cuda.empty_cache()

# ===================== PATHS =====================
TRAIN_IMG_PATH = "/kaggle/input/datasets/rranjithkumariiitk/train-image/train"
DEV_IMG_PATH   = "/kaggle/input/datasets/rranjithkumariiitk/dev-image/dev"
TEST_IMG_PATH  = "/kaggle/input/datasets/rranjithkumariiitk/test-image/test"

train_df = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/train-text/train (1).csv")
dev_df   = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/dev-text/dev (1).csv")
test_df  = pd.read_csv("/kaggle/input/datasets/rranjithkumariiitk/test-dataset/test.csv")

# ===================== LABEL =====================
label_map = {"not-misogyny": 0, "misogyny": 1}
train_df["label"] = train_df["original_labels"].map(label_map)
dev_df["label"]   = dev_df["original_labels"].map(label_map)

# ===================== CLASS WEIGHTS (IMPORTANT) =====================
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
weights = torch.tensor(weights, dtype=torch.float)

# ===================== TOKENIZER =====================
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# ===================== TRANSFORM =====================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ===================== DATASET =====================
class MemeDataset(Dataset):
    def __init__(self, df, img_path, is_test=False):
        self.df = df
        self.img_path = img_path
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["transcriptions"])
        img_id = str(row["image_id"])

        img_path = f"{self.img_path}/{img_id}.jpg"
        if not os.path.exists(img_path):
            img_path = img_path.replace(".jpg", ".png")

        if os.path.exists(img_path):
            image = Image.open(img_path).convert("RGB")
            image = transform(image)
        else:
            image = torch.zeros(3,224,224)

        enc = tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image": image
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.long)

        return item

# ===================== LOADERS =====================
train_loader = DataLoader(MemeDataset(train_df, TRAIN_IMG_PATH), batch_size=1, shuffle=True)
dev_loader   = DataLoader(MemeDataset(dev_df, DEV_IMG_PATH), batch_size=1)
test_loader  = DataLoader(MemeDataset(test_df, TEST_IMG_PATH, True), batch_size=1)

# ===================== MODEL =====================
class MultiModalModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_model = AutoModel.from_pretrained("google/muril-base-cased")

        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Identity()

        # Freeze everything first
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.resnet.parameters():
            p.requires_grad = False

        # 🔥 UNFREEZE LAST LAYER (IMPORTANT)
        for p in self.resnet.layer4.parameters():
            p.requires_grad = True

        self.text_proj  = nn.Linear(768, 256)
        self.image_proj = nn.Linear(2048, 256)

        # 🔥 IMPROVED CLASSIFIER
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, ids, mask, img):

        with torch.no_grad():
            t = self.text_model(input_ids=ids, attention_mask=mask).last_hidden_state[:,0,:]

        r = self.resnet(img)  # partially trainable now

        t = self.text_proj(t)
        i = self.image_proj(r)

        return self.classifier(torch.cat((t,i), dim=1))

# ===================== INIT =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiModalModel().to(device)

criterion = nn.CrossEntropyLoss(weight=weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
scaler = torch.cuda.amp.GradScaler()

# ===================== TRAIN =====================
for epoch in range(8):   # increased epochs
    model.train()
    total_loss = 0

    for b in train_loader:
        ids = b['input_ids'].to(device)
        mask = b['attention_mask'].to(device)
        img = b['image'].to(device)
        labels = b['label'].to(device)

        with torch.cuda.amp.autocast():
            out = model(ids, mask, img)
            loss = criterion(out, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

# ===================== EVAL =====================
model.eval()
preds, true = [], []

with torch.no_grad():
    for b in dev_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)
        true.extend(b['label'].numpy())

print("Prediction classes:", set(preds))
print("F1:", f1_score(true, preds))

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarni

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
  4%|▍         | 4.12M/97.8M [00:00<00:02, 43.0MB/s]
 10%|█         | 10.1M/97.8M [00:00<00:01, 54.6MB/s]
 16%|█▋        | 16.0M/97.8M [00:00<00:01, 57.7MB/s]
 30%|███       | 29.8M/97.8M [00:00<00:00, 91.5MB/s]
 47%|████▋     | 45.5M/97.8M [00:00<00:00, 118MB/s] 
 65%|██████▌   | 63.9M/97.8M [00:00<00:00, 143MB/s]
 84%|████████▍ | 82.2M/97.8M [00:00<00:00, 159MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 108MB/s]
/tmp/ipykernel_55/3037665494.py:155: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_55/3037665494.py:168: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1, Loss: 0.6801879852078855
Epoch 2, Loss: 0.6616558058187365
Epoch 3, Loss: 0.6092758165672422
Epoch 4, Loss: 0.23796768096799498
Epoch 5, Loss: 0.030748078934948353
Epoch 6, Loss: 0.024992867075502544
Epoch 7, Loss: 0.02217986637945728
Epoch 8, Loss: 0.014461724643970797
Prediction classes: {np.int64(0), np.int64(1)}
F1: 0.4444444444444444


In [2]:
# ===================== TEST =====================
model.eval()
preds = []

with torch.no_grad():
    for b in test_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)

# Convert back to labels
reverse_map = {0:"not-misogyny", 1:"misogyny"}
test_df["original_labels"] = [reverse_map[p] for p in preds]

# Save file
test_df.to_csv("submission.csv", index=False)

print("✅ Submission file saved as submission.csv")

✅ Submission file saved as submission.csv


In [3]:
# ===================== TEST =====================
model.eval()
preds = []

with torch.no_grad():
    for b in test_loader:
        out = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['image'].to(device)
        )
        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)

# ✅ Save numeric labels (0 / 1)
test_df["original_labels"] = preds

# Save file
test_df.to_csv("submission.csv", index=False)

print("✅ Submission file saved with 0 (not-misogyny) and 1 (misogyny)")

✅ Submission file saved with 0 (not-misogyny) and 1 (misogyny)
